# Microduck — Phase 1: benchmark of situations

Phase 1 of the work plan (`docs/Piano_lavoro_tesi_Preda.md`, section 5): run the official policy `alpha_walking` (and, for comparison, `rough-walk-g`) in the situations of the benchmark, with episodes at **fixed condition** and episodes with **transitions**, recording falls, velocity error and mean motor power.

| Situation (plan) | Conditions here |
|---|---|
| flat (reference) | `flat` |
| uphill and downhill, also beyond the training maximum | planar-face pyramid walked straight up or down, 0-15° (`face_*`); training pyramid walked down from the top, 3-9° (`pyramid_*`, phase 0) |
| low steps and uneven ground | stairs down and **up** 1-3 cm, grid 0.3-1.5 cm, rubble ±0.2-2.5 cm (phase 0); waves (suggested in section 4.6) |
| slippery floor, below the training minimum (0.7) | friction 0.6 → 0.1 on **both** the feet and the floor |
| payload beyond the ±5% of training | 10-30% of the robot mass **on the centre of mass** (centre of mass unchanged) and 20-30% **offset** forward / back / sideways / up (centre of mass moved) |
| transitions during the episode | flat → downhill → flat, flat → uphill → flat; the floor becomes slippery; a payload appears |
| vibrations (optional in the plan) | not included |

**Product of the phase (plan):** evaluation suite and first table of results. The run also records, **at every control step**, the data of section 4.8 of the plan (observations, activations, action, body motion, feet, torques, scenario, outcome), in the format of section 4.9 (sections 6 and 10 here): they are the data for probes, SAE and recognizer. **Criterion to go on:** at least three situations where the official policy gets measurably worse; only those enter the library (phase 2).

**Simulator.** The same as phase 0 (`fase0_battery.py`), which reproduced the policy cards (32/110 vs 12/110 falls): CPU MuJoCo, robot of `scripts/infer_policy.py`, XL330 servos simulated by bam, 61-D observations, policy at 50 Hz. Phase 1 adds the custom terrains, friction, payload, the events and a heading hold. Two terrains are written here, built as mjlab's heightfields (same contact physics as the other slopes):

- **planar-face pyramid** for the directional slopes: mjlab's pyramid (H·tent(x)·tent(y)) has creases on the centre lines, so a robot walking straight up or down the middle of a "face" straddles a crease with its feet on opposite cross slopes (±5° at a nominal 10°, measured in MuJoCo). Here H·min(tent(x), tent(y)): planar faces;
- **profile strip** for the transitions (flat → slope → flat). The ramp of `slope_terrain.py` is not used: it was built for the roller-skate task (one long tilted box) and made both policies fall already at 2.5°, against 0% on a 5° pyramid face.

**Variability.** The simulation has no noise, so identical starts give identical episodes. Each episode draws a battery voltage (`VIN_RANGE`, as the training randomization), a small offset of the initial joint angles, and the start (random heading on the patches; small heading and lateral jitter on the directional terrains). The draws depend only on condition and episode number: every policy gets the same episodes (paired comparison).

Results go to `MyDrive/microduck_thesis/fase1/<RUN_NAME>/`. **Before you start:** *Runtime → Change runtime type → T4 GPU* (the simulation runs on the CPU; the GPU renders the videos).

## 0. Parameters

The only cell to edit: simulation settings and the **list of conditions** (`CONDITIONS`). Re-run it after every change, then the sections you need.

In [ ]:
# ============================== PHASE 1 PARAMETERS ==============================
# ---- code ----
REPO_URL = "https://github.com/pollen-robotics/microduck_rl"
REPO_REF = "6cd45fc"   # same commit as phase 0 (hostile-terrain branch)
REPO_DIR = "/content/microduck_rl"

# ---- results ----
DRIVE_DIR = "/content/drive/MyDrive/microduck_thesis/fase1"
RUN_NAME = "bench_v1"      # results folder: change it to keep an older run
FORCE_RERUN = False        # False: section 8 is skipped if the run folder already has results
ONLY = None                # e.g. ["flat", "friction_0.3"] to run a subset (names of CONDITIONS)

# ---- policies: Hugging Face (repo, file) or a path ----
POLICIES = {
    "alpha_walking": ("pollen-robotics/microduck-policies", "alpha_walking.onnx"),
    # "rough_walk_g": ("RemiFabre/microduck-rough-walk-g", "policy.onnx"),   # the plan's comparison (doubles the time)
}

# ---- episodes ----
EPISODES_PER_CONDITION = 20
EPISODE_S = 20.0               # default length; a condition can set its own "episode_s"
COMMAND_VX_M_S = 0.3           # constant forward command (as phase 0)
HEADING_HOLD = {"enabled": True, "stiffness": 1.0, "max_rad_s": 1.0}   # yaw-rate command = clip(k x heading error)
SEED = 0
VIN_RANGE = [7.0, 7.8]         # battery voltage drawn per episode [V] (training 6.5-8.2); None = ACTUATOR["vin"]
INIT_JOINT_NOISE_RAD = 0.02    # uniform offset of the initial joint angles
HEADING_JITTER_DEG = 5.0       # directional terrains: start heading +- this
START_LATERAL_JITTER_M = 0.3   # directional terrains: start offset across the walking direction
WARMUP_S = 1.0                 # metrics averaged after this time

# ---- simulation (as phase 0) ----
SCENE_XML = "scene.xml"
PHYSICS_DT_S = 0.005
DECIMATION = 4
SPAWN_HEIGHT_M = 0.125
ACTION_SCALE = 1.0
FALL_TILT_DEG = 70.0
ACTUATOR = {"model": "bam", "kp": 200.0, "vin": 7.4, "max_current_a": 1.75}
PATCH_SIZE_M = 8.0             # patches of the terrain conditions (4 m from the centre to the edge)
BORDER_WIDTH_M = 4.0
WORKERS = 0                    # parallel processes (0 = one per CPU core)
LOG_EVERY_S = 10.0             # a progress line every N simulated seconds of each episode (0 = only the end)

# ---- per-step data (plan sections 4.8-4.9): one .npz per (policy, condition) in RUN_DIR/steps ----
SAVE_STEPS = True              # see section 10 for the size on Drive
SAVE_ACTIVATIONS = True        # the three hidden layers (float16): most of the size
FALL_HORIZON_S = 2.0           # label falls_within_2s: True on the steps less than this before a fall

# ---- videos: first episode of the listed conditions ([] = all) ----
VIDEO = {"enabled": True, "every_n_steps": 2, "width": 640, "height": 480,
         "conditions": ["flat", "stairs_down_2.0cm", "stairs_up_2.0cm", "face_10deg_downhill", "face_10deg_uphill",
                        "friction_0.2", "payload_30pct_com", "payload_30pct_back", "trans_down_10deg",
                        "trans_friction_0.15", "trans_payload_30pct_back"]}

# ---- conditions ----
# terrain kinds: flat | grid (height_cm) | stairs (height_cm, direction "down" from the top or "up" from the
# edge) | rubble (height_cm) | waves (amplitude_cm, peak-to-peak = 2 x amplitude) | slope (deg, training
# pyramid from the top) | face_slope (deg, direction "downhill"/"uphill", planar faces) | strip (segments
# [length_m, slope_deg], walked towards +x). Optional: "friction" (feet AND floor), "payload" {"mass_frac" of the
# robot mass, "offset_m" [x fwd, y left, z up] from the robot's centre of mass, trunk frame}, "event"
# {"type": "friction"/"payload", "t_s", "value"}, "episode_s".
CONDITIONS = []
def add(name, group, terrain, **kw):
    CONDITIONS.append({"name": name, "group": group, "terrain": terrain, **kw})

add("flat", "reference", {"kind": "flat"})
for h in (0.3, 0.6, 0.9, 1.2, 1.5):
    add(f"grid_{h}cm", "uneven", {"kind": "grid", "height_cm": h})
for h in (0.2, 0.75, 1.3, 1.9, 2.5):
    add(f"rubble_{h}cm", "uneven", {"kind": "rubble", "height_cm": h})
for a in (0.5, 1.0, 1.5, 2.0):
    add(f"waves_{a}cm", "uneven", {"kind": "waves", "amplitude_cm": a, "num_waves": 8})
for h in (1.0, 1.5, 2.0, 2.5, 3.0):
    add(f"stairs_down_{h}cm", "steps", {"kind": "stairs", "height_cm": h, "direction": "down"})
    add(f"stairs_up_{h}cm", "steps", {"kind": "stairs", "height_cm": h, "direction": "up"})
for d in (3.0, 5.0, 7.0, 9.0):
    add(f"pyramid_{d}deg", "slope", {"kind": "slope", "deg": d})
for d in (0, 2.5, 5, 7.5, 10, 12.5, 15):
    for direction in ("downhill", "uphill"):
        add(f"face_{d}deg_{direction}", "slope", {"kind": "face_slope", "deg": d, "direction": direction})
for mu in (0.6, 0.5, 0.4, 0.3, 0.2, 0.1):
    add(f"friction_{mu}", "friction", {"kind": "flat"}, friction=mu)
for f in (0.1, 0.2, 0.3):   # mass added on the centre of mass: centre of mass unchanged
    add(f"payload_{int(f * 100)}pct_com", "payload", {"kind": "flat"}, payload={"mass_frac": f, "offset_m": [0, 0, 0]})
for f in (0.2, 0.3):        # mass offset from the centre of mass: centre of mass moved
    for side, off in {"front": [0.05, 0, 0], "back": [-0.05, 0, 0], "side": [0, 0.05, 0], "up": [0, 0, 0.08]}.items():
        add(f"payload_{int(f * 100)}pct_{side}", "payload", {"kind": "flat"}, payload={"mass_frac": f, "offset_m": off})
for d in (5, 10):           # transitions of terrain: flat 1.5 m -> slope 1.5 m -> flat 4 m
    add(f"trans_down_{d}deg", "transition", {"kind": "strip", "segments": [[1.5, 0], [1.5, -d], [4.0, 0]]}, episode_s=30.0)
    add(f"trans_up_{d}deg", "transition", {"kind": "strip", "segments": [[1.5, 0], [1.5, d], [4.0, 0]]}, episode_s=30.0)
for mu in (0.3, 0.15):      # the floor becomes slippery at t = 5 s
    add(f"trans_friction_{mu}", "transition", {"kind": "flat"}, event={"type": "friction", "t_s": 5.0, "value": mu})
for side, off in {"com": [0, 0, 0], "back": [-0.05, 0, 0]}.items():   # a payload appears at t = 5 s
    add(f"trans_payload_30pct_{side}", "transition", {"kind": "flat"},
        payload={"mass_frac": 0.3, "offset_m": off}, event={"type": "payload", "t_s": 5.0})

# ---- "measurably worse than flat" (section 9) ----
WORSE = {"p_value": 0.01, "min_vel_error_increase_m_s": 0.02, "min_power_ratio": 1.10}
# ================================================================================

import os
RUN_DIR = os.path.join(DRIVE_DIR, RUN_NAME)
ROBOT_DIR = os.path.join(REPO_DIR, "src/mjlab_microduck/robot/microduck")
SELECTED = [c for c in CONDITIONS if not ONLY or c["name"] in ONLY]
assert len({c["name"] for c in CONDITIONS}) == len(CONDITIONS), "condition names must be unique"
print(f"{len(SELECTED)} conditions x {EPISODES_PER_CONDITION} episodes x {len(POLICIES)} policies "
      f"= {len(SELECTED) * EPISODES_PER_CONDITION * len(POLICIES)} episodes | results: {RUN_DIR}")

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Installation

Same as phase 0: microduck_rl at `REPO_REF` with the versions of its `uv.lock` (mjlab 1.3.0, MuJoCo 3.10.0, bam).

In [ ]:
import os, subprocess

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
os.environ["UV_HTTP_TIMEOUT"] = "600"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch --all --quiet
!git checkout --quiet {REPO_REF}
print("microduck_rl at", subprocess.check_output(["git", "log", "-1", "--format=%h %ad %s", "--date=short"], text=True))
!uv sync
!uv run python -c "import mujoco, mjlab, bam; print('mujoco', mujoco.__version__, '| CPU cores:', __import__('os').cpu_count())"

## 3. Offscreen rendering (preview and videos)

In [ ]:
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

## 4. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUN_DIR, exist_ok=True)
print("results folder:", RUN_DIR)

## 5. Policies

In [ ]:
from huggingface_hub import hf_hub_download

POLICY_PATHS = {}
for name, source in POLICIES.items():
    if isinstance(source, str):
        POLICY_PATHS[name] = source
    else:
        repo, filename = source
        POLICY_PATHS[name] = hf_hub_download(repo, filename, local_dir=f"/content/policies/{name}")
    print(f"{name:15s} -> {POLICY_PATHS[name]} | found: {os.path.exists(POLICY_PATHS[name])}")

## 6. The benchmark script

`fase1_bench.py` runs every (policy, condition) pair in a separate process (`WORKERS`), all its episodes one after the other. How each part of a condition is built:

| Part | Implementation |
|---|---|
| Robot, servos, observations, control | as phase 0: `scene.xml`, bam XL330 `m6` (gain 200, current limit 1.75 A), 61-D observations, 50 Hz |
| Battery voltage | drawn per episode in `VIN_RANGE`; it sets the voltage of the bam servo model and its torque bound |
| Command | vx = 0.3 m/s; with `HEADING_HOLD` the yaw-rate command is clip(heading error, ±1 rad/s), with the start heading as target (like mjlab's heading controller, an operator correcting with the joystick) |
| Terrains of the patches | one mjlab sub-terrain per condition (8 m patch, flat border); start at the centre with a random heading; stairs `up` start near the edge, facing the top |
| Directional slopes | planar-face pyramid, 40 m: start in the middle of the +x face (10 m from the centre, ~9.5 m from the top and the edge), facing the edge (downhill) or the top (uphill), ± jitter |
| Transitions of terrain | profile strip 7 m × 4 m: start on the first flat, walking towards +x |
| Friction | the sliding coefficient is set to the same value on both feet and on every terrain geom: in a contact MuJoCo takes the larger of the two, so changing one only would not lower it |
| Payload | a non-colliding 2 cm sphere welded to the trunk, at the robot's centre of mass (default pose) + offset. Offset 0: the mass is added on the centre of mass, which does not move. Otherwise the centre of mass moves by offset × m / (M + m) (`com_shift_mm` in the results) |
| Events | at `t_s` the friction is set to `value` (feet and floor), or the payload's mass is switched on (it was 1 mg before) |
| Fall | trunk tilt from vertical > 70° |
| Per-step data | see the next table (`SAVE_STEPS`) |
| Metrics | falls; velocity error \|v_xy − command\| (instantaneous, and averaged over 1 s windows: the tracking without the in-step sway); speed along the robot's forward axis; yaw rate; mechanical motor power Σ\|torque × joint speed\|; after an event, the same metrics from 1 s after it |

**Data recorded at every control step** (plan, section 4.8), one row per step of every episode. The observation is the one the policy received at that step; the state (body, feet, torques) is the one reached after its action (4 physics steps).

| Group (plan) | Fields | Numbers per step | How it is measured here |
|---|---|---|---|
| Policy input | `obs` | 61 | the observation vector (layout in `fields.json`) |
| Activations | `h1`, `h2`, `h3` | 512 + 256 + 128 | forward hooks on the three ELUs of a PyTorch copy of the policy, rebuilt from the ONNX graph as in phase 0; float16. The actions still come from onnxruntime: the largest difference between the two is `torch_onnx_max_diff` of each episode |
| Action | `action` | 14 | policy output; motor targets = default pose + action |
| Body motion | `lin_vel_b`, `ang_vel_b`, `gravity_b` | 9 | trunk velocities in the robot frame, gravity in the robot frame (tilt) |
| Feet | `foot_contact`, `foot_contact_time`, `foot_air_time`, `foot_height` | 8 | contact = a contact between the foot's collision geom and the terrain; times accumulated step by step; height = lowest point of the sole above the terrain under it (vertical ray) |
| Motors | `torque`, `joint_vel`, `power` | 14 + 14 + 1 | actuator torques; joint velocities and power (power is one of the two derived quantities of section 4.8) |
| Scenario | `terrain_kind`, `terrain_value`, `terrain_direction`, `ground_slope_deg`, `ground_cross_slope_deg`, `friction`, `payload_kg`, `com_shift_mm`, `event_active`, `vin_v`, `command` | 15 | the condition, plus what changes during the episode: the slope of the ground actually under the robot (four rays around the trunk: on the transition strip it goes 0 → slope → 0), the current friction and payload. This benchmark has no vibrations and no pushes |
| Outcome | `falls_within_2s` | 1 | computed at the end of the episode |
| Keys | `episode`, `step`, `t_s` | 3 | episode number (row of `bench_results.json`) and time since its start |

**Format** (plan, section 4.9): one compressed NumPy file per block of steps, here one per (policy, condition): `steps/<policy>__<condition>.npz`, plus `steps/fields.json` with meaning and type of every field, the observation layout, the joint order and the terrain codes. Activations in float16 (half the space). The step phase, the other derived quantity of section 4.8, is computed later from contact and air times.

In [ ]:
%%writefile fase1_bench.py
"""Phase 1 (work plan, section 5): benchmark of situations for a walking policy.

Same simulator as the phase 0 battery (fase0_battery.py), which reproduced the policy cards:
CPU MuJoCo, robot of scripts/infer_policy.py, XL330 servos simulated by bam (m6 model), 61-D
observations, policy at 50 Hz. Each CONDITION combines:
  * a terrain: flat, grid, stairs (down from the top or up from the edge), rubble, waves, the training
    pyramid from the top, a pyramid with planar faces walked uphill or downhill, or a strip with a
    profile (flat -> slope -> flat) for the transitions;
  * a friction, set to the SAME value on the feet and on every terrain geom (MuJoCo uses the larger
    of the two coefficients of a contact: changing only one would have no effect);
  * a payload rigidly attached to the trunk: placed exactly on the robot's centre of mass (mass added,
    centre of mass unchanged) or offset from it (centre of mass moved);
  * an event during the episode: the friction drops, or the payload appears (its mass is switched on).
Episodes are paired across policies: same terrain sample, start, heading and battery voltage.

Usage:
    uv run --with "imageio[ffmpeg]" python fase1_bench.py fase1_config.json
    uv run --with "imageio[ffmpeg]" python fase1_bench.py fase1_config.json --preview
"""
import csv
import json
import math
import multiprocessing as mp
import os
import sys
import time
import uuid
from dataclasses import dataclass

import mujoco
import numpy as np
import onnxruntime as ort
from mjlab.terrains import heightfield_terrains as hf_terrains
from mjlab.terrains.heightfield_terrains import (HfPyramidSlopedTerrainCfg, HfRandomUniformTerrainCfg,
                                                 HfWaveTerrainCfg)
from mjlab.terrains.primitive_terrains import (BoxFlatTerrainCfg, BoxPyramidStairsTerrainCfg,
                                               BoxRandomGridTerrainCfg)
from mjlab.terrains.terrain_generator import (SubTerrainCfg, TerrainGenerator, TerrainGeneratorCfg,
                                              TerrainGeometry, TerrainOutput)

# Default pose of scripts/infer_policy.py (STAND keyframe of scene.xml), in actuator order:
# actions are offsets from it, joint observations are relative to it.
DEFAULT_POSE = np.array([
    0.0, -0.0873, -0.4579, -0.0049, 0.4530,    # left leg: hip yaw, hip roll, hip pitch, knee, ankle
    0.3491, 0.3491, 0.0, 0.0,                  # neck pitch, head pitch, head yaw, head roll
    0.0, 0.0873, 0.4579, 0.0049, -0.4530,      # right leg
], dtype=np.float32)
FOOT_GEOMS = ("left_foot_collision", "right_foot_collision")


def log(msg):
    """Print immediately: through a pipe (| grep in the notebook) Python would buffer the output."""
    print(msg, flush=True)


# ---------------------------------------------------------------------------
# Custom heightfield terrains (same construction as mjlab 1.3.0's heightfields)
# ---------------------------------------------------------------------------
def add_hfield(spec, heights_units, size_x, size_y, vertical_scale):
    """Heightfield geom on the terrain body. heights_units has shape (n_y, n_x): MuJoCo rows run
    along y and columns along x, and the n samples of each axis span the whole size."""
    noise = np.rint(heights_units).astype(np.int16)
    lo = int(noise.min())
    span = int(noise.max()) - lo or 1
    height = span * vertical_scale
    norm = (noise - lo) / span
    uid = uuid.uuid4().hex
    field = spec.add_hfield(name=f"hfield_{uid}", size=[size_x / 2, size_y / 2, height, height],
                            nrow=noise.shape[0], ncol=noise.shape[1],
                            userdata=norm.flatten().astype(np.float32).tolist())
    material = hf_terrains.color_by_height(spec, noise, uid, norm)
    geom = spec.body("terrain").add_geom(type=mujoco.mjtGeom.mjGEOM_HFIELD, hfieldname=field.name,
                                         pos=[size_x / 2, size_y / 2, 0.0], material=material)
    return geom, field, height


@dataclass(kw_only=True)
class FlatFacePyramidTerrainCfg(SubTerrainCfg):
    """Pyramid with four PLANAR faces. mjlab's HfPyramidSlopedTerrainCfg is H * tent(x) * tent(y):
    its creases run along the centre lines, so a robot walking straight up or down the middle of a
    "face" straddles a crease, with its feet on opposite cross slopes (about half the nominal slope).
    Here the height is H * min(tent(x), tent(y)): planar faces, creases on the diagonals, and on the
    centre line of a face exactly the nominal slope with no cross slope."""
    slope_deg: float
    platform_width: float = 1.0
    horizontal_scale: float = 0.1
    vertical_scale: float = 0.001

    def function(self, difficulty, spec, rng):
        n = int(self.size[0] / self.horizontal_scale)
        c = int(n / 2)
        h_max = math.tan(math.radians(self.slope_deg)) * self.size[0] / 2 / self.vertical_scale
        tent = (c - np.abs(c - np.arange(n))) / c
        top = h_max * tent[n // 2 - int(self.platform_width / self.horizontal_scale / 2)]
        heights = np.clip(h_max * np.minimum(tent.reshape(-1, 1), tent.reshape(1, -1)), 0, top)
        geom, field, height = add_hfield(spec, heights, self.size[0], self.size[1], self.vertical_scale)
        return TerrainOutput(origin=np.array([self.size[0] / 2, self.size[1] / 2, height]),
                             geometries=[TerrainGeometry(geom=geom, hfield=field)])


@dataclass(kw_only=True)
class ProfileStripTerrainCfg(SubTerrainCfg):
    """A strip whose height changes only along x, built from segments [length_m, slope_deg]
    (positive = uphill when walking towards +x). E.g. [[1.5, 0], [1.5, -10], [4, 0]] is
    flat -> 10 deg downhill -> flat. The size along x must equal the sum of the lengths."""
    segments: list
    horizontal_scale: float = 0.05
    vertical_scale: float = 0.001

    def function(self, difficulty, spec, rng):
        nx = int(round(self.size[0] / self.horizontal_scale)) + 1
        ny = int(round(self.size[1] / self.horizontal_scale)) + 1
        x = np.linspace(0.0, self.size[0], nx)
        z = np.zeros(nx)
        x0, z0 = 0.0, 0.0
        for length, deg in self.segments:
            inside = (x >= x0) & (x <= x0 + length + 1e-9)
            z[inside] = z0 + (x[inside] - x0) * math.tan(math.radians(deg))
            x0, z0 = x0 + length, z0 + length * math.tan(math.radians(deg))
        z -= z.min()
        geom, field, height = add_hfield(spec, np.tile(z / self.vertical_scale, (ny, 1)),
                                         self.size[0], self.size[1], self.vertical_scale)
        return TerrainOutput(origin=np.array([0.0, self.size[1] / 2, 0.0]),
                             geometries=[TerrainGeometry(geom=geom, hfield=field)])


def make_subterrain(t, patch):
    """Terrain entry of a condition -> (mjlab sub-terrain, size of the patch). Obstacle sizes are FIXED
    (difficulty plays no role); shapes of the phase 0 battery / hostile-terrain training menu."""
    kind = t["kind"]
    if kind == "flat":
        return BoxFlatTerrainCfg(proportion=1.0), (patch, patch)
    if kind == "grid":
        h = t["height_cm"] / 100
        return BoxRandomGridTerrainCfg(proportion=1.0, grid_width=0.45, grid_height_range=(h, h),
                                       platform_width=1.0, border_width=0.25), (patch, patch)
    if kind == "stairs":
        h = t["height_cm"] / 100
        return BoxPyramidStairsTerrainCfg(proportion=1.0, step_height_range=(h, h), step_width=0.12,
                                          platform_width=0.8, border_width=0.2), (patch, patch)
    if kind == "rubble":
        h = t["height_cm"] / 100
        return HfRandomUniformTerrainCfg(proportion=1.0, noise_range=(-h, h), noise_step=0.002,
                                         downsampled_scale=0.08, horizontal_scale=0.04, vertical_scale=0.001,
                                         border_width=0.2), (patch, patch)
    if kind == "waves":  # mjlab waves: peak-to-peak height = 2 x amplitude, num_waves along the patch
        a = t["amplitude_cm"] / 100
        return HfWaveTerrainCfg(proportion=1.0, amplitude_range=(a, a), num_waves=t.get("num_waves", 4),
                                horizontal_scale=0.05, vertical_scale=0.001), (patch, patch)
    if kind == "slope":  # training pyramid (with its creases), walked down from the top platform
        s = math.tan(math.radians(t["deg"]))
        return HfPyramidSlopedTerrainCfg(proportion=1.0, slope_range=(s, s), platform_width=0.8,
                                         horizontal_scale=0.1, vertical_scale=0.001), (patch, patch)
    if kind == "face_slope":
        size = t.get("size_m", 40.0)
        return FlatFacePyramidTerrainCfg(proportion=1.0, slope_deg=t["deg"], platform_width=1.0), (size, size)
    if kind == "strip":
        length = sum(s[0] for s in t["segments"])
        return ProfileStripTerrainCfg(proportion=1.0, segments=t["segments"]), (length, t.get("width_m", 4.0))
    raise SystemExit(f"unknown terrain kind {kind!r}")


# ---------------------------------------------------------------------------
# Scene: robot + terrain + friction + payload
# ---------------------------------------------------------------------------
def robot_com_in_trunk(conf):
    """Centre of mass of the whole robot in the trunk frame, in the default pose."""
    spec = mujoco.MjSpec.from_file(os.path.join(conf["robot_dir"], conf["scene_xml"]))
    model = spec.compile()
    data = mujoco.MjData(model)
    jids = model.actuator_trnid[:, 0]
    data.qpos[model.jnt_qposadr[jids]] = DEFAULT_POSE
    mujoco.mj_forward(model, data)
    trunk = model.body("trunk_base").id
    rot = data.xmat[trunk].reshape(3, 3)
    com_world = data.subtree_com[trunk]
    return rot.T @ (com_world - data.xpos[trunk]), float(model.body_subtreemass[trunk])


def build_model(conf, cond, terrain_seed, com_trunk, robot_mass):
    spec = mujoco.MjSpec.from_file(os.path.join(conf["robot_dir"], conf["scene_xml"]))
    floor = spec.geom("floor")
    if floor is not None:
        spec.delete(floor)  # replaced by the generated terrain (patch + flat border at z = 0)
    sub, size = make_subterrain(cond["terrain"], conf["patch_size_m"])
    gen = TerrainGenerator(TerrainGeneratorCfg(
        size=size, border_width=conf["border_width_m"], num_rows=1, num_cols=1, curriculum=False,
        difficulty_range=(1.0, 1.0), sub_terrains={cond["name"]: sub}, add_lights=False, seed=terrain_seed))
    gen.compile(spec)

    payload = cond.get("payload")
    if payload:
        # Rigid payload welded to the trunk: a non-colliding sphere at the robot's centre of mass + offset
        # (offset 0: mass added, centre of mass unchanged). If it appears during the episode, it starts
        # with a negligible mass and the event switches its mass on.
        mass = payload["mass_frac"] * robot_mass
        pos = com_trunk + np.asarray(payload.get("offset_m", [0, 0, 0]), dtype=float)
        body = spec.body("trunk_base").add_body(name="payload", pos=pos.tolist())
        start_mass = 1e-6 if cond.get("event", {}).get("type") == "payload" else mass
        body.add_geom(type=mujoco.mjtGeom.mjGEOM_SPHERE, size=[0.02, 0, 0], mass=start_mass,
                      contype=0, conaffinity=0, group=1,   # group 1: visible, ignored by the ground rays
                      rgba=[0.9, 0.15, 0.15, 1.0])

    act = conf["actuator"]
    if act["model"] == "bam":
        # As scripts/testbench_sim2real.py: torque motors driven by bam's MujocoController; the force
        # range (voltage bound) is set per episode from the battery voltage; joint friction and damping
        # are written by bam at every physics step.
        for a in spec.actuators:
            a.set_to_motor()
            a.forcelimited = True
            a.gear = [1.0, 0, 0, 0, 0, 0]
        for j in spec.joints:
            if j.type == mujoco.mjtJoint.mjJNT_HINGE:
                j.damping = np.zeros_like(j.damping) if np.ndim(j.damping) else 0.0  # array in newer MuJoCo
                j.frictionloss = 0.0
    model = spec.compile()
    model.opt.timestep = conf["physics_dt_s"]
    if act["model"] == "position" and act.get("max_current_a"):
        from bam.model import load_model
        kt = load_model(motor_name="xl330", model="m6").kt.value
        model.actuator_forcerange[:] = [-kt * act["max_current_a"], kt * act["max_current_a"]]
        model.actuator_forcelimited[:] = 1
    return model, gen.terrain_origins[0, 0].copy(), size


def friction_geoms(model):
    """Ids of the geoms whose friction a friction condition sets: both feet and every terrain geom."""
    terrain = model.body("terrain").id
    ids = [g for g in range(model.ngeom) if model.geom_bodyid[g] == terrain]
    feet = [model.geom(n).id for n in FOOT_GEOMS]
    return ids + feet, feet


def set_friction(model, mu):
    ids, _ = friction_geoms(model)
    model.geom_friction[ids, 0] = mu  # sliding coefficient; torsional and rolling unchanged


def ground_height(model, data, x, y, radius=0.1):
    """Highest terrain point within `radius` of (x, y), by vertical rays (robot moved out of the way)."""
    best = -np.inf
    geomid = np.zeros(1, dtype=np.int32)
    for dx in np.linspace(-radius, radius, 5):
        for dy in np.linspace(-radius, radius, 5):
            d = mujoco.mj_ray(model, data, np.array([x + dx, y + dy, 50.0]), np.array([0.0, 0.0, -1.0]),
                              None, 1, -1, geomid)
            if d >= 0:
                best = max(best, 50.0 - d)
    return float(best)


def start_pose(cond, origin, size, rng, conf):
    """(x, y, yaw) of the start; the heading hold keeps the start yaw."""
    t = cond["terrain"]
    jitter = math.radians(conf["heading_jitter_deg"])
    lat = conf["start_lateral_jitter_m"]
    kind, direction = t["kind"], t.get("direction")
    if kind == "face_slope":   # middle of the +x face (planar), facing the edge (downhill) or the top
        d = t.get("start_m", size[0] / 4)
        yaw = (0.0 if direction == "downhill" else math.pi) + rng.uniform(-jitter, jitter)
        return origin[0] + d, origin[1] + rng.uniform(-lat, lat), yaw
    if kind == "stairs" and direction == "up":   # near the edge of the patch, facing the top
        return origin[0] + size[0] / 2 - 0.35, origin[1] + rng.uniform(-lat, lat), math.pi + rng.uniform(-jitter, jitter)
    if kind == "strip":        # on the first segment, walking towards +x along the profile
        return origin[0] + t.get("start_m", 0.6), origin[1] + rng.uniform(-lat, lat), rng.uniform(-jitter, jitter)
    return origin[0], origin[1], rng.uniform(-math.pi, math.pi)   # centre of the patch, random heading


# ---------------------------------------------------------------------------
# Robot and policy
# ---------------------------------------------------------------------------
class Robot:
    def __init__(self, model, conf):
        self.model, self.conf = model, conf
        self.data = mujoco.MjData(model)
        self.trunk = model.body("trunk_base").id
        fj = model.joint("trunk_base_freejoint").id
        self.free_qpos, self.free_dof = model.jnt_qposadr[fj], model.jnt_dofadr[fj]
        self.gyro_adr = model.sensor_adr[model.sensor("imu_ang_vel").id]
        jids = model.actuator_trnid[:, 0]
        self.qpos_idx = model.jnt_qposadr[jids]
        self.dof_idx = model.jnt_dofadr[jids]
        if model.nu != len(DEFAULT_POSE):
            raise SystemExit(f"expected {len(DEFAULT_POSE)} actuators, found {model.nu}")
        self.ctrl = self.bam = None
        act = conf["actuator"]
        if act["model"] == "bam":
            from bam.model import load_model
            from bam.mujoco import MujocoController
            self.bam = load_model(motor_name="xl330", model="m6")
            self.bam.actuator.kp = act["kp"]
            if hasattr(self.bam.actuator, "max_current"):
                self.bam.actuator.max_current = act.get("max_current_a")
            self.kt_over_r = self.bam.kt.value / self.bam.R.value
            names = [model.actuator(i).name for i in range(model.nu)]
            self.ctrl = MujocoController(self.bam, names, model, self.data)
        # feet and terrain, for contacts, foot heights and the slope under the robot
        terrain = model.body("terrain").id
        self.is_terrain = model.geom_bodyid == terrain
        groups = set(model.geom_group[self.is_terrain].tolist())
        robot_groups = set(model.geom_group[~self.is_terrain].tolist())
        if len(groups) != 1 or groups & robot_groups:
            raise SystemExit(f"terrain geoms must be alone in one geom group (terrain {groups}, others {robot_groups})")
        self.ray_group = np.zeros(6, dtype=np.uint8)
        self.ray_group[groups.pop()] = 1          # rays hit the terrain only
        self.feet = [model.geom(n).id for n in FOOT_GEOMS]
        self.foot_verts = [model.mesh_vert[model.mesh_vertadr[i]:model.mesh_vertadr[i] + model.mesh_vertnum[i]].copy()
                           for i in model.geom_dataid[self.feet]]   # sole meshes, in the geom frame
        self._down = np.array([0.0, 0.0, -1.0])
        self._gid = np.zeros(1, dtype=np.int32)
        self.frictionloss0 = model.dof_frictionloss.copy()
        self.damping0 = model.dof_damping.copy()
        self.geom_friction0 = model.geom_friction.copy()
        self.body_mass0 = model.body_mass.copy()
        self.body_inertia0 = model.body_inertia.copy()

    def reset(self, x, y, z, yaw, vin, joint_noise):
        m, d = self.model, self.data
        m.dof_frictionloss[:] = self.frictionloss0
        m.dof_damping[:] = self.damping0
        m.geom_friction[:] = self.geom_friction0
        m.body_mass[:] = self.body_mass0
        m.body_inertia[:] = self.body_inertia0
        mujoco.mj_setConst(m, d)
        if self.bam is not None:
            self.bam.actuator.vin = vin
            m.actuator_forcerange[:] = [-vin * self.kt_over_r, vin * self.kt_over_r]
        mujoco.mj_resetData(m, d)
        a = self.free_qpos
        d.qpos[a:a + 3] = [x, y, z]
        d.qpos[a + 3:a + 7] = [math.cos(yaw / 2), 0.0, 0.0, math.sin(yaw / 2)]
        d.qpos[self.qpos_idx] = DEFAULT_POSE + joint_noise
        if self.ctrl is None:
            d.ctrl[:] = DEFAULT_POSE
        mujoco.mj_forward(m, d)
        if self.ctrl is not None:
            self.ctrl.last_ts = d.time
            self.ctrl.reset(d.qpos)
            self.ctrl.q_target[:] = DEFAULT_POSE

    def to_body(self, vec):
        q = self.data.xquat[self.trunk]
        w, v = q[0], q[1:4]
        t = np.cross(v, vec) * 2
        return vec - w * t + np.cross(v, t)

    def gravity_b(self):
        return self.to_body(np.array([0.0, 0.0, -1.0]))

    def lin_vel_b(self):
        return self.to_body(self.data.qvel[self.free_dof:self.free_dof + 3].copy())

    def yaw(self):
        w, x, y, z = self.data.xquat[self.trunk]
        return math.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))

    def ang_vel_b(self):
        return self.data.qvel[self.free_dof + 3:self.free_dof + 6].copy()   # free joint: already in the body frame

    def _ground(self, x, y, z):
        """Terrain height under (x, y), by a ray from height z (terrain geoms only)."""
        dist = mujoco.mj_ray(self.model, self.data, np.array([x, y, z]), self._down, self.ray_group, 1, -1, self._gid)
        return z - dist if dist >= 0 else np.nan

    def feet_state(self):
        """Per foot: contact with the terrain, and height of the lowest point of the sole above the ground [m]."""
        d = self.data
        n = d.ncon
        g1, g2 = d.contact.geom1[:n], d.contact.geom2[:n]
        contact = np.array([bool(np.any(((g1 == f) & self.is_terrain[g2]) | ((g2 == f) & self.is_terrain[g1])))
                            for f in self.feet])
        height = np.empty(2)
        for i, (f, verts) in enumerate(zip(self.feet, self.foot_verts)):
            w = d.geom_xpos[f] + verts @ d.geom_xmat[f].reshape(3, 3).T
            p = w[np.argmin(w[:, 2])]
            height[i] = p[2] - self._ground(p[0], p[1], p[2] + 0.03)
        return contact, height

    def ground_slope(self, r=0.1):
        """Slope of the ground under the trunk [deg]: along the robot's heading (+ = uphill) and across it
        (+ = rising to the robot's left), from four rays 2r apart. The true situation at every step."""
        x, y, z = self.data.xpos[self.trunk]
        z0 = z - 0.03
        c, s = math.cos(self.yaw()), math.sin(self.yaw())
        hf, hb = self._ground(x + r * c, y + r * s, z0), self._ground(x - r * c, y - r * s, z0)
        hl, hr = self._ground(x - r * s, y + r * c, z0), self._ground(x + r * s, y - r * c, z0)
        return math.degrees(math.atan2(hf - hb, 2 * r)), math.degrees(math.atan2(hl - hr, 2 * r))

    def observation(self, last_action, command):
        d = self.data
        return np.concatenate([
            d.sensordata[self.gyro_adr:self.gyro_adr + 3],
            self.gravity_b(),
            d.qpos[self.qpos_idx] - DEFAULT_POSE,
            d.qvel[self.dof_idx],
            last_action,
            command,
        ]).astype(np.float32)

    def step(self, target):
        m, d = self.model, self.data
        if self.ctrl is not None:
            self.ctrl.q_target[:] = target
            self.ctrl.update()
        else:
            d.ctrl[:] = target
        mujoco.mj_step(m, d)
        return float(np.abs(d.actuator_force * d.qvel[self.dof_idx]).sum())


class Policy:
    def __init__(self, path):
        self.sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        self.inp = self.sess.get_inputs()[0].name
        n = self.sess.get_inputs()[0].shape[-1]
        if n != 61:
            raise SystemExit(f"{path}: the policy takes {n} observations, this benchmark builds the 61-D layout")

    def __call__(self, obs):
        return self.sess.run(None, {self.inp: obs[None]})[0][0].astype(np.float32)


class Activations:
    """The same policy as a PyTorch nn.Sequential (rebuilt from the ONNX graph, as fase0_torch.py, where
    it matched onnxruntime within 1e-5), with forward hooks on the three ELUs (plan sections 4.5, 4.10).
    It only READS the activations: the actions still come from onnxruntime, and the largest difference
    between the two outputs is recorded as a check."""

    def __init__(self, path):
        import onnx
        import torch
        from onnx import numpy_helper
        torch.set_num_threads(1)   # one process per (policy, condition): no thread oversubscription
        self.torch = torch

        class Elementwise(torch.nn.Module):
            def __init__(self, op, c):
                super().__init__()
                self.op = op
                self.register_buffer("c", torch.as_tensor(np.asarray(c, dtype=np.float32)))

            def forward(self, x):
                return {"Sub": torch.sub, "Add": torch.add, "Mul": torch.mul, "Div": torch.div}[self.op](x, self.c)

        g = onnx.load(path).graph
        const = {i.name: numpy_helper.to_array(i).copy() for i in g.initializer}
        folding = {"Add": np.add, "Sub": np.subtract, "Mul": np.multiply, "Div": np.divide}
        mods = []
        for n in g.node:
            at = {a.name: onnx.helper.get_attribute_value(a) for a in n.attribute}
            op, ins = n.op_type, list(n.input)
            if op == "Constant":
                const[n.output[0]] = numpy_helper.to_array(at["value"])
                continue
            if all(i in const for i in ins if i):   # node on constants only: fold it now
                v = [const[i] for i in ins if i]
                if op in folding:
                    const[n.output[0]] = folding[op](*v)
                elif op == "Sqrt":
                    const[n.output[0]] = np.sqrt(v[0])
                elif op in ("Identity", "Cast"):
                    const[n.output[0]] = v[0]
                else:
                    raise ValueError(f"unsupported ONNX op on constants: {op}")
                continue
            if op in ("Gemm", "MatMul"):
                w = const[ins[1]] if (op == "Gemm" and at.get("transB", 0)) else const[ins[1]].T
                lin = torch.nn.Linear(w.shape[1], w.shape[0], bias=op == "Gemm" and len(ins) > 2 and ins[2] in const)
                with torch.no_grad():
                    lin.weight.copy_(torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32)))
                    if lin.bias is not None:
                        lin.bias.copy_(torch.from_numpy(const[ins[2]].astype(np.float32).reshape(-1)))
                mods.append(lin)
            elif op in ("Add", "Sub", "Mul", "Div"):
                c = const[ins[1]]
                if op == "Add" and mods and isinstance(mods[-1], torch.nn.Linear) and mods[-1].bias is None:
                    mods[-1].bias = torch.nn.Parameter(torch.from_numpy(c.astype(np.float32).reshape(-1)))
                else:
                    mods.append(Elementwise(op, c))
            elif op == "Elu":
                mods.append(torch.nn.ELU(alpha=at.get("alpha", 1.0)))
            elif op == "Clip":
                lo = float(const[ins[1]]) if len(ins) > 1 and ins[1] else at.get("min", -np.inf)
                hi = float(const[ins[2]]) if len(ins) > 2 and ins[2] else at.get("max", np.inf)
                mods.append(torch.nn.Hardtanh(lo, hi))   # = clamp(x, lo, hi)
            elif op in ("Identity", "Cast", "Flatten", "Reshape", "Squeeze", "Unsqueeze"):
                continue
            else:
                raise ValueError(f"unsupported ONNX op: {op}")
        self.model = torch.nn.Sequential(*mods).eval()
        elus = [m for m in self.model if isinstance(m, torch.nn.ELU)]
        if len(elus) != 3:
            raise SystemExit(f"{path}: expected 3 hidden layers (ELU), found {len(elus)}")
        self.out = {}
        for k, m in enumerate(elus):
            m.register_forward_hook(lambda mod, inp, out, k=k: self.out.__setitem__(f"h{k + 1}", out[0].numpy().copy()))

    def __call__(self, obs):
        """-> (h1, h2, h3, PyTorch action)"""
        with self.torch.no_grad():
            y = self.model(self.torch.from_numpy(obs[None]))[0].numpy()
        return self.out["h1"], self.out["h2"], self.out["h3"], y


class Video:
    def __init__(self, model, conf):
        v = conf["video"]
        self.renderer = mujoco.Renderer(model, v["height"], v["width"])
        self.cam = mujoco.MjvCamera()
        self.cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
        self.cam.trackbodyid = model.body("trunk_base").id
        self.cam.distance, self.cam.elevation, self.cam.azimuth = 0.9, -20.0, 135.0
        self.every = v["every_n_steps"]
        self.frames = []

    def capture(self, data, step):
        if step % self.every == 0:
            self.renderer.update_scene(data, camera=self.cam)
            self.frames.append(self.renderer.render().copy())

    def save(self, path, fps):
        import imageio
        imageio.mimsave(path, self.frames, fps=fps)
        self.renderer.close()


# ---------------------------------------------------------------------------
# One episode
# ---------------------------------------------------------------------------
TERRAIN_KINDS = ["flat", "grid", "stairs", "rubble", "waves", "slope", "face_slope", "strip"]
DIRECTIONS = {None: 0, "down": 1, "downhill": 1, "up": 2, "uphill": 2}


def terrain_value(t):
    """The size of the terrain condition: obstacle height [cm], wave amplitude [cm] or slope [deg]."""
    for key in ("height_cm", "amplitude_cm", "deg"):
        if key in t:
            return float(t[key])
    if t["kind"] == "strip":
        return float(max((abs(s[1]) for s in t["segments"]), default=0.0))
    return 0.0


def episode(robot, policy, conf, cond, origin, size, rng, robot_mass, video=None, tag="",
            acts=None, steps=None, com_trunk=None):
    """Run one episode. If `steps` is a dict of lists, append one row per control step to it
    (plan section 4.8); `acts` (Activations) adds the three hidden layers."""
    m, d = robot.model, robot.data
    x, y, yaw0 = start_pose(cond, origin, size, rng, conf)
    vin = rng.uniform(*conf["vin_range"]) if conf.get("vin_range") else conf["actuator"]["vin"]
    noise = rng.uniform(-conf["init_joint_noise_rad"], conf["init_joint_noise_rad"], 14)
    robot.reset(x, y, 100.0, yaw0, vin, noise)          # robot out of the way of the ground rays
    # highest ground point under the feet; on the long slopes a smaller radius, otherwise the uphill
    # side of the circle would lift the start (10 cm x tan 15 deg = 2.7 cm above the surface)
    radius = 0.05 if cond["terrain"]["kind"] in ("face_slope", "strip") else 0.1
    z0 = ground_height(m, d, x, y, radius) + conf["spawn_height_m"]
    robot.reset(x, y, z0, yaw0, vin, noise)
    if cond.get("friction") is not None:
        set_friction(m, cond["friction"])

    event = cond.get("event")
    decimation = conf["decimation"]
    control_dt = decimation * conf["physics_dt_s"]
    episode_s = cond.get("episode_s", conf["episode_s"])
    n_steps = int(round(episode_s / control_dt))
    event_step = int(round(event["t_s"] / control_dt)) if event else None
    warmup = int(round(conf["warmup_s"] / control_dt))
    log_every = int(round(conf.get("log_every_s", 0) / control_dt))
    hold = conf["heading_hold"]
    cos_fall = math.cos(math.radians(conf["fall_tilt_deg"]))

    command = np.zeros(13, dtype=np.float32)
    command[0] = conf["command_vx_m_s"]
    last_action = np.zeros(14, dtype=np.float32)
    start = d.qpos[robot.free_qpos:robot.free_qpos + 2].copy()
    rec = {k: [] for k in ("vx", "vy", "wz", "power", "tilt")}
    outcome, step, max_tilt, max_torch_diff = "completed", 0, 0.0, 0.0

    # scenario of the episode (plan section 4.8): what can change during the episode is updated below
    t = cond["terrain"]
    friction_now = cond.get("friction") if cond.get("friction") is not None else float(robot.geom_friction0[robot.feet[0], 0])
    payload = cond.get("payload")
    payload_kg = payload["mass_frac"] * robot_mass if payload else 0.0
    payload_shift = (np.asarray(payload.get("offset_m", [0, 0, 0])) * 1000 * payload_kg / (robot_mass + payload_kg)
                     if payload else np.zeros(3))
    payload_on = bool(payload) and not (event and event["type"] == "payload")
    contact_time, air_time = np.zeros(2), np.zeros(2)
    first_row = len(steps.get("step", [])) if steps is not None else 0

    for step in range(n_steps):
        if event and step == event_step:
            if event["type"] == "friction":
                set_friction(m, event["value"])
                friction_now = event["value"]
            elif event["type"] == "payload":
                pid = m.body("payload").id
                mass = cond["payload"]["mass_frac"] * robot_mass
                # mass and inertia enter the dynamics at every step (mj_setConst would overwrite the state)
                m.body_mass[pid] = mass
                m.body_inertia[pid] = 0.4 * mass * 0.02 ** 2   # solid sphere, 2 cm radius
                payload_on = True
            log(f"      {tag} t={step * control_dt:.1f} s: event {event['type']} applied")
        if hold["enabled"]:  # mjlab-style heading controller: yaw-rate command from the heading error
            err = math.atan2(math.sin(yaw0 - robot.yaw()), math.cos(yaw0 - robot.yaw()))
            command[2] = float(np.clip(hold["stiffness"] * err, -hold["max_rad_s"], hold["max_rad_s"]))
        obs = robot.observation(last_action, command)
        action = policy(obs)
        last_action = action
        target = DEFAULT_POSE + conf["action_scale"] * action
        p = 0.0
        for _ in range(decimation):
            p += robot.step(target)
        if video is not None:
            video.capture(d, step)
        if not np.all(np.isfinite(d.qpos)):
            outcome = "nan"
            break
        v = robot.lin_vel_b()
        up = -robot.gravity_b()[2]
        tilt = math.degrees(math.acos(float(np.clip(up, -1.0, 1.0))))
        max_tilt = max(max_tilt, tilt)
        for k, val in (("vx", v[0]), ("vy", v[1]), ("wz", d.sensordata[robot.gyro_adr + 2]),
                       ("power", p / decimation), ("tilt", tilt)):
            rec[k].append(float(val))
        if steps is not None:
            # one row per control step: the observation the policy received, its action, and the state
            # reached after the action (4 physics steps)
            contact, foot_height = robot.feet_state()
            contact_time = np.where(contact, contact_time + control_dt, 0.0)
            air_time = np.where(contact, 0.0, air_time + control_dt)
            slope_along, slope_across = robot.ground_slope()
            row = {
                "episode": steps["_episode"], "step": step, "t_s": (step + 1) * control_dt,
                "obs": obs, "action": action,
                "lin_vel_b": v, "ang_vel_b": robot.ang_vel_b(), "gravity_b": robot.gravity_b(),
                "foot_contact": contact, "foot_contact_time": contact_time.copy(), "foot_air_time": air_time.copy(),
                "foot_height": foot_height, "torque": d.actuator_force.copy(), "joint_vel": d.qvel[robot.dof_idx].copy(),
                "power": p / decimation, "command": command[:3].copy(),
                "terrain_kind": TERRAIN_KINDS.index(t["kind"]), "terrain_value": terrain_value(t),
                "terrain_direction": DIRECTIONS[t.get("direction")],
                "ground_slope_deg": slope_along, "ground_cross_slope_deg": slope_across,
                "friction": friction_now, "payload_kg": payload_kg if payload_on else 0.0,
                "com_shift_mm": payload_shift if payload_on else np.zeros(3),
                "event_active": bool(event) and step >= event_step, "vin_v": vin,
            }
            if acts is not None:
                h1, h2, h3, y = acts(obs)
                row.update(h1=h1, h2=h2, h3=h3)
                max_torch_diff = max(max_torch_diff, float(np.abs(y - action).max()))
            for key, val in row.items():
                steps.setdefault(key, []).append(val)
        if up < cos_fall:
            outcome = "fell"
            break
        if log_every and (step + 1) % log_every == 0:
            log(f"      {tag} sim {(step + 1) * control_dt:5.1f}/{episode_s:.0f} s | tilt {tilt:4.1f} deg | "
                f"vx {v[0]:+.2f} m/s | {np.linalg.norm(d.qpos[robot.free_qpos:robot.free_qpos + 2] - start):.2f} m")

    duration = (step + 1) * control_dt
    if steps is not None:
        # outcome label (plan 4.8): True on the rows within `fall_horizon_s` before a fall
        horizon = int(round(conf.get("fall_horizon_s", 2.0) / control_dt))
        n_rows = len(steps.get("step", [])) - first_row
        rows_steps = np.asarray(steps.get("step", [])[first_row:])
        label = (rows_steps > step - horizon) if outcome == "fell" else np.zeros(n_rows, dtype=bool)
        steps.setdefault("falls_within_2s", []).extend(label.tolist())
    r = {k: np.asarray(v) for k, v in rec.items()}
    sl = slice(warmup, None)
    cmd = np.array([conf["command_vx_m_s"], 0.0])
    err = np.hypot(r["vx"][sl] - cmd[0], r["vy"][sl] - cmd[1]) if len(r["vx"]) > warmup else np.array([])
    win = max(1, int(round(1.0 / control_dt)))   # 1 s windows: tracking without the in-step oscillation
    n_win = len(r["vx"][sl]) // win
    err_1s = (np.hypot(r["vx"][sl][:n_win * win].reshape(n_win, win).mean(1) - cmd[0],
                       r["vy"][sl][:n_win * win].reshape(n_win, win).mean(1) - cmd[1]) if n_win else np.array([]))
    pos = d.qpos[robot.free_qpos:robot.free_qpos + 2]
    mean = lambda a: float(a.mean()) if len(a) else None  # noqa: E731
    res = {
        "outcome": outcome, "duration_s": duration,
        "fall_time_s": duration if outcome == "fell" else None,
        "vel_error_m_s": mean(err), "vel_error_1s_m_s": mean(err_1s),
        "path_speed_m_s": mean(r["vx"][sl]), "yaw_rate_rad_s": mean(r["wz"][sl]),
        "power_w": mean(r["power"][sl]), "max_tilt_deg": max_tilt,
        "distance_m": float(np.linalg.norm(pos - start)),
        "start_x": float(start[0] - origin[0]), "start_y": float(start[1] - origin[1]),
        "end_x": float(pos[0] - origin[0]), "end_y": float(pos[1] - origin[1]),
        "heading_deg": math.degrees(yaw0), "vin_v": float(vin),
    }
    if acts is not None:
        res["torch_onnx_max_diff"] = max_torch_diff
    if event:
        res["fell_after_event"] = outcome == "fell" and step >= event_step
        after = slice(event_step + win, None)   # 1 s after the event
        res["vel_error_after_event_m_s"] = mean(np.hypot(r["vx"][after] - cmd[0], r["vy"][after] - cmd[1]))
        res["power_after_event_w"] = mean(r["power"][after])
    log(f"      {tag} -> {outcome.upper()} at {duration:.1f}/{episode_s:.0f} s | max tilt {max_tilt:.0f} deg | "
        f"vel err {res['vel_error_1s_m_s'] if res['vel_error_1s_m_s'] is None else round(res['vel_error_1s_m_s'], 3)} m/s")
    return res


# ---------------------------------------------------------------------------
# One (policy, condition) task: all its episodes
# ---------------------------------------------------------------------------
def run_task(args):
    conf, pname, ppath, ci = args
    cond = conf["conditions"][ci]
    t0 = time.time()
    com_trunk, robot_mass = robot_com_in_trunk(conf)
    model, origin, size = build_model(conf, cond, conf["seed"] + ci, com_trunk, robot_mass)
    robot, policy = Robot(model, conf), Policy(ppath)
    save_steps = conf.get("save_steps", False)
    acts = Activations(ppath) if save_steps and conf.get("save_activations", True) else None
    steps = {} if save_steps else None
    log(f"[{pname}] condition {ci + 1}/{len(conf['conditions'])}: {cond['name']} ({cond['group']}) - "
        f"{conf['episodes_per_condition']} episodes")
    episodes = []
    vconf = conf["video"]
    for k in range(conf["episodes_per_condition"]):
        rng = np.random.default_rng([conf["seed"], ci, k])   # same draws for every policy (paired)
        video = None
        if vconf["enabled"] and k == 0 and (not vconf["conditions"] or cond["name"] in vconf["conditions"]):
            try:
                video = Video(model, conf)
            except Exception as e:  # noqa: BLE001  rendering is optional
                log(f"[WARNING] no video, the renderer could not start ({e}); is MUJOCO_GL=egl set?")
        if steps is not None:
            steps["_episode"] = k
        ep = episode(robot, policy, conf, cond, origin, size, rng, robot_mass, video,
                     tag=f"[{pname}] {cond['name']} ep {k + 1}/{conf['episodes_per_condition']}",
                     acts=acts, steps=steps, com_trunk=com_trunk)
        ep.update({"policy": pname, "condition": cond["name"], "group": cond["group"], "episode": k})
        episodes.append(ep)
        if video is not None:
            os.makedirs(os.path.join(conf["out_dir"], "videos"), exist_ok=True)
            fps = 1.0 / (conf["decimation"] * conf["physics_dt_s"] * vconf["every_n_steps"])
            video.save(os.path.join(conf["out_dir"], "videos", f"{pname}__{cond['name']}.mp4"), fps)
    if steps is not None:
        path = save_step_file(conf, pname, cond, steps)
        log(f"[{pname}] {cond['name']}: {len(steps['step'])} steps saved to {os.path.basename(path)} "
            f"({os.path.getsize(path) / 1e6:.1f} MB)")
    falls = sum(e["outcome"] == "fell" for e in episodes)
    log(f"[{pname}] {cond['name']} done: falls {falls}/{len(episodes)} ({time.time() - t0:.0f} s)")
    return episodes


# Per-step fields (plan section 4.8): dtype in the .npz files and meaning
STEP_FIELDS = {
    "episode": ("int16", "episode number in this condition (key into 'episodes' of bench_results.json)"),
    "step": ("int16", "control step since the start of the episode (50 Hz)"),
    "t_s": ("float32", "time since the start of the episode, at the end of this step [s]"),
    "obs": ("float32", "the 61 observations fed to the policy at this step (layout in fields.json)"),
    "h1": ("float16", "activation of hidden layer 1 (after ELU), 512 numbers"),
    "h2": ("float16", "activation of hidden layer 2 (after ELU), 256 numbers"),
    "h3": ("float16", "activation of hidden layer 3 (after ELU), 128 numbers"),
    "action": ("float32", "the 14 outputs of the policy: motor targets = default pose + action"),
    "lin_vel_b": ("float32", "trunk linear velocity in the robot frame, after the step [m/s]"),
    "ang_vel_b": ("float32", "trunk angular velocity in the robot frame, after the step [rad/s]"),
    "gravity_b": ("float32", "gravity direction in the robot frame (tilt), after the step"),
    "foot_contact": ("bool", "left / right foot touching the terrain"),
    "foot_contact_time": ("float32", "time each foot has been on the ground [s] (0 in the air)"),
    "foot_air_time": ("float32", "time each foot has been in the air [s] (0 on the ground)"),
    "foot_height": ("float32", "height of the lowest point of each sole above the terrain under it [m]"),
    "torque": ("float32", "torques of the 14 motors at the last physics step [Nm]"),
    "joint_vel": ("float32", "velocities of the 14 joints [rad/s]"),
    "power": ("float32", "mechanical motor power, sum of |torque x joint velocity|, mean over the step [W]"),
    "command": ("float32", "velocity command fed to the policy: vx, vy [m/s], yaw rate [rad/s] (heading hold)"),
    "terrain_kind": ("int8", "index in terrain_kinds (fields.json)"),
    "terrain_value": ("float32", "size of the terrain condition: obstacle height or wave amplitude [cm], or slope [deg]"),
    "terrain_direction": ("int8", "0 none, 1 down / downhill, 2 up / uphill"),
    "ground_slope_deg": ("float32", "measured slope of the ground under the trunk along the heading (+ = uphill) [deg]"),
    "ground_cross_slope_deg": ("float32", "measured slope of the ground under the trunk across the heading (+ = rising to the left) [deg]"),
    "friction": ("float32", "current friction coefficient of feet and floor"),
    "payload_kg": ("float32", "payload carried at this step [kg] (0 before it appears)"),
    "com_shift_mm": ("float32", "shift of the robot's centre of mass caused by the payload, x fwd / y left / z up [mm]"),
    "event_active": ("bool", "True from the event of a transition condition onwards"),
    "vin_v": ("float32", "battery voltage of the episode [V]"),
    "falls_within_2s": ("bool", "True if the episode ends with a fall within the next fall_horizon_s (2 s)"),
}
OBS_LAYOUT = {"base_ang_vel": [0, 3], "projected_gravity": [3, 6], "joint_pos_rel": [6, 20], "joint_vel": [20, 34],
              "last_action": [34, 48], "twist_command": [48, 51], "head_command": [51, 55], "body_command": [55, 61]}


def save_step_file(conf, pname, cond, steps):
    """One compressed .npz per (policy, condition): a block of steps, one row per control step (plan 4.9)."""
    out = os.path.join(conf["out_dir"], "steps")
    os.makedirs(out, exist_ok=True)
    arrays = {k: np.asarray(v, dtype=STEP_FIELDS[k][0]) for k, v in steps.items() if not k.startswith("_")}
    path = os.path.join(out, f"{pname}__{cond['name']}.npz")
    np.savez_compressed(path, **arrays)
    return path


def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval of a proportion, in percent."""
    if n == 0:
        return None
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return [100.0 * max(0.0, centre - half), 100.0 * min(1.0, centre + half)]


def summarize(episodes, conf, robot_mass, com_trunk):
    out = []
    for pname in conf["policies"]:
        for cond in conf["conditions"]:
            eps = [e for e in episodes if e["policy"] == pname and e["condition"] == cond["name"]]
            if not eps:
                continue
            n, falls = len(eps), sum(e["outcome"] == "fell" for e in eps)
            row = {"policy": pname, "condition": cond["name"], "group": cond["group"], "episodes": n,
                   "falls": falls, "falls_pct": 100.0 * falls / n, "falls_ci95_pct": wilson_ci(falls, n),
                   "nan": sum(e["outcome"] == "nan" for e in eps),
                   "mean_duration_s": float(np.mean([e["duration_s"] for e in eps]))}
            # walking metrics over the COMPLETED episodes only: in a fall they measure the fall itself
            done = [e for e in eps if e["outcome"] == "completed"]
            row["completed"] = len(done)
            for key in ("vel_error_m_s", "vel_error_1s_m_s", "path_speed_m_s", "yaw_rate_rad_s", "power_w",
                        "max_tilt_deg", "distance_m", "vel_error_after_event_m_s", "power_after_event_w"):
                vals = np.array([e[key] for e in done if e.get(key) is not None], dtype=float)
                if len(vals):
                    row[key] = float(vals.mean())
                    row[key + "_sem"] = float(vals.std(ddof=1) / math.sqrt(len(vals))) if len(vals) > 1 else None
            if cond.get("event"):
                row["falls_after_event"] = sum(bool(e.get("fell_after_event")) for e in eps)
            if cond.get("payload"):
                p = cond["payload"]
                mass = p["mass_frac"] * robot_mass
                off = np.asarray(p.get("offset_m", [0, 0, 0]), dtype=float)
                row["payload_kg"] = mass
                row["com_shift_mm"] = (1000 * off * mass / (robot_mass + mass)).round(1).tolist()
            out.append(row)
    return out


def preview(conf):
    """One still per condition with the robot at its start (payload in red)."""
    import imageio
    com_trunk, robot_mass = robot_com_in_trunk(conf)
    tiles = []
    for ci, cond in enumerate(conf["conditions"]):
        model, origin, size = build_model(conf, cond, conf["seed"] + ci, com_trunk, robot_mass)
        robot = Robot(model, conf)
        rng = np.random.default_rng([conf["seed"], ci, 0])
        x, y, yaw = start_pose(cond, origin, size, rng, conf)
        robot.reset(x, y, 100.0, yaw, conf["actuator"]["vin"], np.zeros(14))
        radius = 0.05 if cond["terrain"]["kind"] in ("face_slope", "strip") else 0.1
        z = ground_height(model, robot.data, x, y, radius) + conf["spawn_height_m"]
        robot.reset(x, y, z, yaw, conf["actuator"]["vin"], np.zeros(14))
        r = mujoco.Renderer(model, 180, 240)
        cam = mujoco.MjvCamera()
        cam.lookat[:] = [x, y, z - 0.1]
        cam.distance, cam.elevation, cam.azimuth = 2.0, -25.0, math.degrees(yaw) + 150.0
        r.update_scene(robot.data, camera=cam)
        tiles.append(r.render().copy())
        r.close()
        log(f"[preview] {ci + 1:2d} {cond['name']}: start ({x - origin[0]:+.2f}, {y - origin[1]:+.2f}) m, "
            f"heading {math.degrees(yaw):+.0f} deg, trunk z {z:.3f} m")
    cols = 6
    blank = np.zeros_like(tiles[0])
    rows = [np.concatenate(tiles[i:i + cols] + [blank] * (cols - len(tiles[i:i + cols])), axis=1)
            for i in range(0, len(tiles), cols)]
    path = os.path.join(conf["out_dir"], "conditions_preview.png")
    imageio.imwrite(path, np.concatenate(rows, axis=0))
    log(f"saved {path} (conditions in order, {cols} per row)")


def main(conf):
    os.makedirs(conf["out_dir"], exist_ok=True)
    t0 = time.time()
    com_trunk, robot_mass = robot_com_in_trunk(conf)
    log(f"robot mass {robot_mass:.3f} kg, centre of mass in the trunk frame {np.round(com_trunk * 1000, 1)} mm")
    names = [c["name"] for c in conf["conditions"]]
    if len(names) != len(set(names)):
        raise SystemExit("condition names must be unique")
    tasks = [(conf, p, path, ci) for p, path in conf["policies"].items() for ci in range(len(conf["conditions"]))]
    if conf.get("save_steps"):
        os.makedirs(os.path.join(conf["out_dir"], "steps"), exist_ok=True)
        with open(os.path.join(conf["out_dir"], "steps", "fields.json"), "w") as f:
            json.dump({"fields": {k: {"dtype": v[0], "description": v[1]} for k, v in STEP_FIELDS.items()},
                       "obs_layout": OBS_LAYOUT, "terrain_kinds": TERRAIN_KINDS,
                       "joint_order": ["left_hip_yaw", "left_hip_roll", "left_hip_pitch", "left_knee", "left_ankle",
                                       "neck_pitch", "head_pitch", "head_yaw", "head_roll", "right_hip_yaw",
                                       "right_hip_roll", "right_hip_pitch", "right_knee", "right_ankle"],
                       "default_pose": DEFAULT_POSE.tolist(), "control_dt_s": conf["decimation"] * conf["physics_dt_s"],
                       "fall_horizon_s": conf.get("fall_horizon_s", 2.0),
                       "files": "steps/<policy>__<condition>.npz, one row per control step of every episode"},
                      f, indent=1)
    workers = conf.get("workers") or os.cpu_count() or 1
    log(f"{len(tasks)} tasks (policy x condition), {conf['episodes_per_condition']} episodes each, {workers} worker(s)")
    episodes = []
    if workers == 1:
        for t in tasks:
            episodes += run_task(t)
    else:
        with mp.Pool(workers) as pool:
            for eps in pool.imap_unordered(run_task, tasks):
                episodes += eps
    summary = summarize(episodes, conf, robot_mass, com_trunk)
    out = {"config": conf, "robot_mass_kg": robot_mass, "robot_com_trunk_m": com_trunk.tolist(),
           "summary": summary, "episodes": episodes, "runtime_s": time.time() - t0,
           "versions": {"mujoco": mujoco.__version__, "onnxruntime": ort.__version__}}
    with open(os.path.join(conf["out_dir"], "bench_results.json"), "w") as f:
        json.dump(out, f, indent=1)
    keys = sorted({k for e in episodes for k in e})
    with open(os.path.join(conf["out_dir"], "bench_episodes.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(episodes)
    log(f"done in {time.time() - t0:.0f} s -> {conf['out_dir']}")


if __name__ == "__main__":
    with open(sys.argv[1]) as f:
        cfg = json.load(f)
    preview(cfg) if "--preview" in sys.argv else main(cfg)

## 7. Preview of the conditions

One still per condition with the robot at its start (`conditions_preview.png`, in the order of `CONDITIONS`, six per row; the payload is the red sphere). Check the terrains and the starts before running.

In [ ]:
import json
from IPython.display import Image, display

def bench_config():
    return {
        "robot_dir": ROBOT_DIR, "scene_xml": SCENE_XML, "policies": POLICY_PATHS, "conditions": SELECTED,
        "episodes_per_condition": EPISODES_PER_CONDITION, "episode_s": EPISODE_S, "command_vx_m_s": COMMAND_VX_M_S,
        "heading_hold": HEADING_HOLD, "seed": SEED, "vin_range": VIN_RANGE, "init_joint_noise_rad": INIT_JOINT_NOISE_RAD,
        "heading_jitter_deg": HEADING_JITTER_DEG, "start_lateral_jitter_m": START_LATERAL_JITTER_M,
        "warmup_s": WARMUP_S, "physics_dt_s": PHYSICS_DT_S, "decimation": DECIMATION, "spawn_height_m": SPAWN_HEIGHT_M,
        "action_scale": ACTION_SCALE, "fall_tilt_deg": FALL_TILT_DEG, "actuator": ACTUATOR,
        "patch_size_m": PATCH_SIZE_M, "border_width_m": BORDER_WIDTH_M, "workers": WORKERS,
        "log_every_s": LOG_EVERY_S, "video": VIDEO, "out_dir": RUN_DIR,
        "save_steps": SAVE_STEPS, "save_activations": SAVE_ACTIVATIONS, "fall_horizon_s": FALL_HORIZON_S,
    }

with open("fase1_config.json", "w") as f:
    json.dump(bench_config(), f, indent=1)
!uv run --with "imageio[ffmpeg]" python -u fase1_bench.py fase1_config.json --preview 2>&1 | grep -v "Terrain generation took"
display(Image(os.path.join(RUN_DIR, "conditions_preview.png")))

## 8. Run the benchmark

Skipped if `bench_results.json` is already in the run folder (set `FORCE_RERUN = True` or change `RUN_NAME`). The log is also saved to `bench_log.txt`. With the default list (68 conditions × 20 episodes, 20-30 s each) the table alone took 32 minutes per policy on Colab's CPUs; recording the per-step data makes the simulation about 2.3 times slower (roughly 1-1.5 hours per policy) and writes about 2 kB per step, about 2 GB per policy on Drive (about 0.5 GB with `SAVE_ACTIVATIONS = False`). Use `ONLY` or fewer `EPISODES_PER_CONDITION` for a quick test.

In [ ]:
with open("fase1_config.json", "w") as f:
    json.dump(bench_config(), f, indent=1)
if os.path.exists(os.path.join(RUN_DIR, "bench_results.json")) and not FORCE_RERUN:
    print("benchmark already run in", RUN_DIR, "- set FORCE_RERUN = True to run it again")
else:
    !uv run --with "imageio[ffmpeg]" python -u fase1_bench.py fase1_config.json 2>&1 | grep --line-buffered -v "Terrain generation took" | tee {RUN_DIR}/bench_log.txt

## 9. Results

One row per (policy, condition): falls with the 95% Wilson interval, velocity error (1 s windows), speed, yaw rate, power, and the comparison with `flat` of the same policy. Velocity error, speed, yaw rate and power are averaged over the **completed** episodes only (`completed` column): during a fall they would measure the fall itself; a condition where every episode falls has no walking metrics, only falls. A condition is **worse** than flat when at least one of:
- **falls**: the 95% intervals do not overlap;
- **velocity error**: higher, Welch test p < `WORSE["p_value"]` and at least `min_vel_error_increase_m_s` more;
- **power**: higher, p < `p_value` and at least `min_power_ratio` × flat.

For the conditions with an event, `falls after event` counts the falls after the change, and the `after event` columns average from 1 s after it. For the payloads, `com shift` is how much the payload moves the robot's centre of mass (x forward, y left, z up).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

with open(os.path.join(RUN_DIR, "bench_results.json")) as f:
    res = json.load(f)
print(f"robot mass {res['robot_mass_kg']:.3f} kg | runtime {res['runtime_s'] / 60:.0f} min")
ep = pd.DataFrame(res["episodes"])
S = pd.DataFrame(res["summary"])

def compare(row):
    ref = S[(S.policy == row.policy) & (S.condition == "flat")]
    if ref.empty or row.condition == "flat":
        return pd.Series({"worse_falls": False, "worse_vel": False, "worse_power": False})
    ref = ref.iloc[0]
    # walking metrics compared on the completed episodes only (in a fall they measure the fall)
    a = ep[(ep.policy == row.policy) & (ep.condition == row.condition) & (ep.outcome == "completed")]
    b = ep[(ep.policy == row.policy) & (ep.condition == "flat") & (ep.outcome == "completed")]
    worse_falls = row.falls_ci95_pct[0] > ref.falls_ci95_pct[1]
    def worse(col, ok):
        x, y = a[col].dropna(), b[col].dropna()
        if len(x) < 2 or len(y) < 2:
            return False
        p = stats.ttest_ind(x, y, equal_var=False).pvalue
        return bool(p < WORSE["p_value"] and ok(x.mean(), y.mean()))
    return pd.Series({
        "worse_falls": bool(worse_falls),
        "worse_vel": worse("vel_error_1s_m_s", lambda m, r: m - r >= WORSE["min_vel_error_increase_m_s"]),
        "worse_power": worse("power_w", lambda m, r: m >= WORSE["min_power_ratio"] * r),
    })

S = pd.concat([S, S.apply(compare, axis=1)], axis=1)
S["worse"] = S[["worse_falls", "worse_vel", "worse_power"]].any(axis=1)
fmt = lambda v, n=3: "" if v is None or (isinstance(v, float) and np.isnan(v)) else round(v, n)
table = pd.DataFrame({
    "policy": S.policy, "group": S.group, "condition": S.condition,
    "falls": S.falls.astype(str) + "/" + S.episodes.astype(str),
    "falls 95% CI (%)": S.falls_ci95_pct.apply(lambda c: f"{c[0]:.0f}-{c[1]:.0f}"),
    "completed": S.completed,
    "vel err 1s (m/s)": S.vel_error_1s_m_s.apply(fmt),
    "speed (m/s)": S.path_speed_m_s.apply(fmt),
    "yaw rate (rad/s)": S.yaw_rate_rad_s.apply(fmt),
    "power (W)": S.power_w.apply(fmt),
    "falls after event": S.get("falls_after_event", pd.Series(np.nan, index=S.index)).apply(lambda v: "" if pd.isna(v) else int(v)),
    "vel err after event": S.get("vel_error_after_event_m_s", pd.Series(np.nan, index=S.index)).apply(fmt),
    "com shift (mm)": S.get("com_shift_mm", pd.Series(np.nan, index=S.index)).apply(lambda v: v if isinstance(v, list) else ""),
    "worse than flat": S.apply(lambda r: ", ".join(k for k in ("falls", "vel", "power") if r[f"worse_{k}"]), axis=1),
})
pd.set_option("display.max_rows", 500)
display(table.set_index(["policy", "group", "condition"]))
table.to_csv(os.path.join(RUN_DIR, "bench_table.csv"), index=False)

print("\nSituations where the policy is measurably worse than flat (plan criterion: at least 3):")
for p in S.policy.unique():
    w = S[(S.policy == p) & S.worse]
    groups = sorted(set(w.group) - {"reference"})
    print(f"  {p}: {len(groups)} situation(s): " + "; ".join(f"{g} ({', '.join(w[w.group == g].condition)})" for g in groups))
    print(f"    criterion {'MET' if len(groups) >= 3 else 'NOT met'}")

Videos of the first episode of the conditions listed in `VIDEO`.

In [ ]:
import glob
from IPython.display import HTML, Video

for path in sorted(glob.glob(os.path.join(RUN_DIR, "videos", "*.mp4"))):
    policy, cond = os.path.basename(path)[:-4].split("__")
    e = ep[(ep.policy == policy) & (ep.condition == cond) & (ep.episode == 0)]
    info = f"{e.iloc[0].outcome} after {e.iloc[0].duration_s:.1f} s" if len(e) else ""
    display(HTML(f"<b>{policy}</b> - {cond} - {info}"))
    display(Video(path, embed=True, width=480))

## 10. The per-step data

Lists the files in `RUN_DIR/steps` with their size, then loads one condition as an example: shape, type and meaning of every field, and three checks: the PyTorch copy used for the activations matches the ONNX actions, the slope measured under the robot follows the terrain, and the fall labels. Every file loads on its own with `np.load(path)`, also in a new session after mounting Drive.

In [ ]:
import glob

steps_dir = os.path.join(RUN_DIR, "steps")
files = sorted(glob.glob(os.path.join(steps_dir, "*.npz")))
total_mb = sum(os.path.getsize(f) for f in files) / 1e6
print(f"{len(files)} files, {total_mb / 1000:.2f} GB in {steps_dir}")
fields = json.load(open(os.path.join(steps_dir, "fields.json")))

EXAMPLE = "trans_down_5deg"   # any condition of the run
policy = next(iter(POLICIES))
data = np.load(os.path.join(steps_dir, f"{policy}__{EXAMPLE}.npz"))
print(f"{policy} / {EXAMPLE}: {len(data['step'])} steps, episodes {np.unique(data['episode']).tolist()}")
display(pd.DataFrame([{"field": k, "shape": data[k].shape, "dtype": str(data[k].dtype),
                       "meaning": fields["fields"][k]["description"]} for k in data.files]).set_index("field"))

e = ep[(ep.policy == policy) & (ep.condition == EXAMPLE)]
if "torch_onnx_max_diff" in e:
    print(f"PyTorch vs ONNX actions, largest difference over the episodes: {e.torch_onnx_max_diff.max():.1e}")
first = data["episode"] == 0
print("episode 0, slope measured under the robot every 2 s [deg]:",
      np.round(data["ground_slope_deg"][first][::100].astype(float), 1).tolist())
print("steps labelled 'falls within 2 s':", int(data["falls_within_2s"].sum()), "of", len(data["step"]))

## Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| `ModuleNotFoundError: bam` or `mjlab` | section 2 not run in this session | re-run section 2, then 6-8 |
| Preview or videos fail with an EGL / OpenGL error | section 3 not run, or a CPU-only runtime | run section 3; or set `VIDEO["enabled"] = False` (the numbers do not need rendering) |
| `NaN` episodes | the simulation diverged | check `ACTUATOR`, `PHYSICS_DT_S` and extreme conditions |
| The run is too slow | many conditions × episodes on 2 CPU cores | fewer `EPISODES_PER_CONDITION`, a subset with `ONLY`, or a runtime with more cores (`WORKERS = 0` uses them all) |
| A condition never shows up in the results | it was not in `ONLY`, or its task failed (see the log) | check `bench_log.txt` |
| Drive full, or the run slower than before | the per-step data | `SAVE_ACTIVATIONS = False` (much smaller) or `SAVE_STEPS = False` for a run of the table only |